# Day 5 (Tue Aug 18) — Let's build GPT (1h56m) — the main event

He types EVERYTHING live from an empty notebook — so this starter is just the data + targets. Work in tandem: pause when he names a thing, build it, unpause to compare. This notebook = dev scratchpad (his gpt-dev); consolidate into gpt.py at the end (CS336 tests import a .py).

**arc:** read data → encode/decode → batches → bigram baseline (val ~2.5) → the mathematical trick → single head → multi-head → feedforward → blocks + residuals + layernorm → scale up
**targets:** bigram baseline val ~2.5 · final GPT (n_embd 384, 6 layers, 6 heads, block 256, ~10M params) **val ~1.48** on shakespeare chars
**reuse from my week:** embeddings, Linear, cross-entropy, training loop, lr decay, eval-mode discipline, param-count alarm, shape walks — all of it appears again here

In [367]:
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)

In [ ]:
BATCH_SIZE = 4 # how many independent sequences will we process in parallel?
BLOCK_SIZE = 8 # what is the maximum context length for predictions?
TRAINING_STEPS = 1000 

In [368]:
# input.txt already downloaded (tiny shakespeare, 1,115,394 chars) — no wget needed
with open('input.txt', 'r') as f:
    text = f.read()
print(len(text))
print(text[:200])

1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you


In [369]:
unique_text = sorted(set(text))
vocab_size = len(unique_text)
itos= {ch: i for ch, i in enumerate(unique_text)}
stoi= {i: ch for ch, i in enumerate(unique_text)}

encode = lambda s:[stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])


In [370]:
data = torch.tensor(encode(text), dtype=torch.long)

# Split validation, and train data
n = int(0.9 * len(data))
train_data = data[:n] 
val_data = data[n:] 

In [372]:
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - BLOCK_SIZE, (BATCH_SIZE,))
    x = torch.stack([data[i:i+BLOCK_SIZE] for i in ix]) # 
    y = torch.stack([data[i+1:i+BLOCK_SIZE+1] for i in ix])
    return x, y

In [373]:
xb, yb = get_batch('test')

print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----')

for b in range(BATCH_SIZE): # batch dimension
    for t in range(BLOCK_SIZE): # time dimension
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")

inputs:
torch.Size([4, 8])
tensor([[ 6,  1, 52, 53, 58,  1, 58, 47],
        [ 6,  1, 54, 50, 39, 52, 58, 43],
        [ 1, 58, 46, 47, 57,  1, 50, 47],
        [ 0, 32, 46, 43, 56, 43,  1, 42]])
targets:
torch.Size([4, 8])
tensor([[ 1, 52, 53, 58,  1, 58, 47, 50],
        [ 1, 54, 50, 39, 52, 58, 43, 58],
        [58, 46, 47, 57,  1, 50, 47, 60],
        [32, 46, 43, 56, 43,  1, 42, 53]])
----
when input is [6] the target: 1
when input is [6, 1] the target: 52
when input is [6, 1, 52] the target: 53
when input is [6, 1, 52, 53] the target: 58
when input is [6, 1, 52, 53, 58] the target: 1
when input is [6, 1, 52, 53, 58, 1] the target: 58
when input is [6, 1, 52, 53, 58, 1, 58] the target: 47
when input is [6, 1, 52, 53, 58, 1, 58, 47] the target: 50
when input is [6] the target: 1
when input is [6, 1] the target: 54
when input is [6, 1, 54] the target: 50
when input is [6, 1, 54, 50] the target: 39
when input is [6, 1, 54, 50, 39] the target: 52
when input is [6, 1, 54, 50, 39, 52] t

In [ ]:
class BigramLanguageModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
    
    def forward(self, x, target=None):
        logit = self.token_embedding_table(x) # Batch, Time, Channels
        # Pytorch cross_entropy doc wants it in Batch, Channels, Time format
        B, T, C = logit.shape

        if target == None:
            loss = None
        else:
            logit = logit.view(B*T, C)
            target = target.view(B*T)
            loss = F.cross_entropy(logit, target)
        return logit, loss

    def generate(self, x, max_new_tokens):
        for _ in range(max_new_tokens):
            # Call forwardb using self
            logit, _ = self(x)
            # We care about starting from the last one
            logit = logit[:, -1, :] # (B, T, C) → (B, C)
            prob = F.softmax(logit, -1)
            sample = torch.multinomial(prob, num_samples=1)
            x = torch.cat((x, sample), dim=-1)
        return x

In [ ]:
bi = BigramLanguageModel()    

x = encode("ROMEO:")
idx = torch.tensor([x])
idx
out = bi.generate(idx, 200)[0].tolist()
decode(out)

"ROMEO:HOMHukRuaRJKXAYtXzfJ:HEPiu--sDioi;ILCo3pHNTmDwJsfheKRxZCFs\nlZJ XQc?:s:HEzEnXalEPklcPU cL'DpdLCafBheHd:.?nA!LgjgL&jcM-sZW!HFsWt'rv.x;z&hPyd'XHOxnAcj.erRjcLxxVFSrOpW.oeGJ Ai$yi&!mRfeopx,f.e..iBgxVrl-VTh"

In [ ]:
# training using AdamW optimizer
optimizer = torch.optim.AdamW(bi.parameters(), lr=1e-3)

for i in ran
